# Fetching data at warp speed

In [3]:
# 1. Install dependencies
!pip install dukascopy-python pandas pyarrow joblib -q

import os
import time
import pandas as pd
import dukascopy_python as dp
from datetime import datetime
from pathlib import Path
from joblib import Parallel, delayed

# --- CONFIGURATION ---
# NOTE: If you want to keep files permanently, mount Google Drive:
from google.colab import drive
drive.mount('/content/drive')
OUT_DIR = Path("/content/drive/MyDrive/GITHUB-COPILOT/stk-mat2011/data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

PAIRS = ["EUR/NOK", "EUR/SEK"]
MONTHS = [
    "202501", "202502", "202503", "202504", "202505", "202506",
    "202507", "202508", "202509", "202510", "202511", "202512",
    "202601", "202602", "202603"
]

# PARALLELISM: Use -1 to use all cores, or 4-8 to avoid getting rate-limited
N_JOBS = 4 

# --- HELPER FUNCTIONS ---
def get_month_range(yyyymm: str):
    year, month = int(yyyymm[:4]), int(yyyymm[4:6])
    start = datetime(year, month, 1)
    end = datetime(year + 1, 1, 1) if month == 12 else datetime(year, month + 1, 1)
    return start, end

def process_job(pair, yyyymm, compression="snappy", max_retries=3):
    """Function to be executed in parallel with existence check and retries"""
    symbol = pair.replace("/", "").lower()
    
    # 1. EXISTENCE CHECK: Define file paths first
    out_name_bid = f"{symbol}_dukascopy_bid_{yyyymm}.parquet"
    out_name_ask = f"{symbol}_dukascopy_ask_{yyyymm}.parquet"
    out_path_bid = OUT_DIR / out_name_bid
    out_path_ask = OUT_DIR / out_name_ask

    # If both files already exist, skip the download completely
    if out_path_bid.exists() and out_path_ask.exists():
        return f"⏭️ Skipped (Already Exists): {pair} {yyyymm}"

    start, end = get_month_range(yyyymm)
    
    # 2. RETRY LOGIC: Handle Dukascopy connection drops
    for attempt in range(max_retries):
        try:
            df = dp.fetch(
                instrument=pair,
                interval=dp.INTERVAL_TICK,
                offer_side=dp.OFFER_SIDE_BID, 
                start=start,
                end=end,
            )

            if df is None or len(df) == 0:
                return f"⚠️ Empty: {pair} {yyyymm}"

            results_stats = []
            for side, price_col, vol_col in [("bid", "bidPrice", "bidVolume"), ("ask", "askPrice", "askVolume")]:
                out_name = out_name_bid if side == "bid" else out_name_ask
                out_path = out_path_bid if side == "bid" else out_path_ask
                
                pd.DataFrame({
                    "datetime": df.index,
                    "symbol": symbol.upper(),
                    "price_type": side.upper(),
                    "price": df[price_col].values,
                    "volume": df[vol_col].values,
                }).to_parquet(out_path, engine="pyarrow", compression=compression, index=False)
                
                results_stats.append(f"{out_name} ({len(df):,} rows)")
                
            return f"✅ Success: {pair} {yyyymm} -> " + " & ".join(results_stats)
        
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(5) # Wait 5 seconds before retrying
                continue
            return f"❌ FAILED: {pair} {yyyymm} | Error after {max_retries} attempts: {str(e)}"

# --- EXECUTION ---
jobs = [(p, m) for p in PAIRS for m in MONTHS]
print(f"🚀 Starting Parallel Download of {len(jobs)} jobs using {N_JOBS} workers...\n")

t0 = time.time()
results = Parallel(n_jobs=N_JOBS, backend="threading")(
    delayed(process_job)(p, m) for p, m in jobs
)

for r in results:
    print(r)

elapsed = time.time() - t0
print(f"\n✨ All done! Finished in {elapsed:.1f}s")
print(f"📁 Files saved to: {OUT_DIR.absolute()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🚀 Starting Parallel Download of 30 jobs using 4 workers...



INFO:DUKASCRIPT:current timestamp :2025-04-01T06:55:05.977000
INFO:DUKASCRIPT:current timestamp :2025-03-03T07:02:15.790000
INFO:DUKASCRIPT:current timestamp :2025-01-02T08:45:43.678000
INFO:DUKASCRIPT:current timestamp :2025-02-03T01:31:42.744000
INFO:DUKASCRIPT:current timestamp :2025-02-03T03:37:33.166000
INFO:DUKASCRIPT:current timestamp :2025-01-02T15:29:06.974000
INFO:DUKASCRIPT:current timestamp :2025-04-01T15:38:08.027000
INFO:DUKASCRIPT:current timestamp :2025-03-03T15:50:41.856000
INFO:DUKASCRIPT:current timestamp :2025-01-02T21:35:25.041000
INFO:DUKASCRIPT:current timestamp :2025-02-03T06:22:39.108000
INFO:DUKASCRIPT:current timestamp :2025-04-02T00:46:30.713000
INFO:DUKASCRIPT:current timestamp :2025-01-03T08:32:07.299000
INFO:DUKASCRIPT:current timestamp :2025-02-03T11:24:53.484000
INFO:DUKASCRIPT:current timestamp :2025-03-03T20:32:33.308000
INFO:DUKASCRIPT:current timestamp :2025-04-02T03:25:17.696000
INFO:DUKASCRIPT:current timestamp :2025-02-03T15:38:21.660000
INFO:DUK

✅ Success: EUR/NOK 202501 -> eurnok_dukascopy_bid_202501.parquet (2,753,923 rows) & eurnok_dukascopy_ask_202501.parquet (2,753,923 rows)
✅ Success: EUR/NOK 202502 -> eurnok_dukascopy_bid_202502.parquet (2,292,096 rows) & eurnok_dukascopy_ask_202502.parquet (2,292,096 rows)
✅ Success: EUR/NOK 202503 -> eurnok_dukascopy_bid_202503.parquet (2,400,671 rows) & eurnok_dukascopy_ask_202503.parquet (2,400,671 rows)
✅ Success: EUR/NOK 202504 -> eurnok_dukascopy_bid_202504.parquet (7,145,420 rows) & eurnok_dukascopy_ask_202504.parquet (7,145,420 rows)
✅ Success: EUR/NOK 202505 -> eurnok_dukascopy_bid_202505.parquet (4,063,539 rows) & eurnok_dukascopy_ask_202505.parquet (4,063,539 rows)
✅ Success: EUR/NOK 202506 -> eurnok_dukascopy_bid_202506.parquet (2,475,496 rows) & eurnok_dukascopy_ask_202506.parquet (2,475,496 rows)
✅ Success: EUR/NOK 202507 -> eurnok_dukascopy_bid_202507.parquet (2,272,827 rows) & eurnok_dukascopy_ask_202507.parquet (2,272,827 rows)
✅ Success: EUR/NOK 202508 -> eurnok_dukas